<a href="https://colab.research.google.com/github/PeroronShine/education_fefu_2/blob/main/cubernetic/%D0%BA%D0%B8%D0%B1%D0%B5%D1%80%D0%BD%D0%B5%D1%82%D0%B8%D0%BA%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

КИБЕРНЕТИКА
______________________________

Задача, написать программный код, который будет выполнять данные алгоритмы, можно по отдельности каждую программу и вычислить избыточность и среднее число символов в коде для каждого алгоритма

На вввод мы подаем ( вводим с клав) два символа (буквы англ алфавита) и вероятность для каждого символа (помним что сумма всех вероятностей должны быть равна 1), на выходе должны получить КОД ( последовательность из 0 и 1 для каждой буквы)

In [ ]:
import pandas as pd
import itertools
from math import log2

class ShannonFanoCombination:
    def __init__(self, symbols, probabilities):
        self.symbols = symbols
        self.original_probabilities = probabilities.copy()
        self.codes = {}

    def build_codes(self, symbol_list, probabilities_dict, current_code=""):
        """Рекурсивное построение кодов Шеннона-Фано"""
        if len(symbol_list) == 1:
            self.codes[symbol_list[0]] = current_code
            return

        sorted_symbols = sorted(symbol_list,
                              key=lambda x: probabilities_dict[x],
                              reverse=True)

        # Находим точку разделения с минимальной разницей сумм вероятностей
        total_prob = sum(probabilities_dict[s] for s in sorted_symbols)
        half_prob = total_prob / 2

        current_sum = 0
        split_index = 0

        for i, symbol in enumerate(sorted_symbols):
            current_sum += probabilities_dict[symbol]
            if abs(current_sum - half_prob) <= abs((current_sum - probabilities_dict[symbol]) - half_prob):
                split_index = i + 1
                break

        left_group = sorted_symbols[:split_index]
        right_group = sorted_symbols[split_index:]

        self.build_codes(left_group, probabilities_dict, current_code + "0")
        self.build_codes(right_group, probabilities_dict, current_code + "1")

    def encode_combinations(self, k_length):
        """Кодирование комбинаций длины K"""
        combinations = list(itertools.product(self.symbols, repeat=k_length))

        comb_probs = {}
        for comb in combinations:
            prob = 1.0
            for symbol in comb:
                prob *= self.original_probabilities[symbol]
            comb_probs[comb] = prob

        sorted_combinations = sorted(comb_probs.keys(),
                                   key=lambda x: comb_probs[x],
                                   reverse=True)

        # Строим коды для комбинаций
        self.codes = {}
        self.build_codes(sorted_combinations, comb_probs)

        return comb_probs

    def get_results_table(self, k_length):
        """Создание таблицы результатов для комбинаций"""
        comb_probs = self.encode_combinations(k_length)

        results = []
        for comb in sorted(comb_probs.keys(), key=lambda x: comb_probs[x], reverse=True):
            prob = comb_probs[comb]
            code = self.codes[comb]
            code_length = len(code)
            comb_str = ''.join(comb)

            results.append({
                'Комбинация': comb_str,
                'Вероятность': f"{prob:.6f}",
                'Код': code,
                'Длина кода': code_length
            })

        df = pd.DataFrame(results)
        return df, comb_probs

    def calculate_statistics(self, comb_probs, k_length):
        """Вычисление статистических характеристик"""
        avg_comb_length = sum(comb_probs[comb] * len(self.codes[comb])
                            for comb in comb_probs)

        avg_symbol_length = avg_comb_length / k_length

        symbol_entropy = -sum(p * log2(p)
                            for p in self.original_probabilities.values() if p > 0)

        comb_entropy = -sum(prob * log2(prob)
                          for prob in comb_probs.values() if prob > 0)

        redundancy_comb = avg_comb_length - comb_entropy
        redundancy_symbol = avg_symbol_length - symbol_entropy

        efficiency_comb = (comb_entropy / avg_comb_length) * 100 if avg_comb_length > 0 else 0
        efficiency_symbol = (symbol_entropy / avg_symbol_length) * 100 if avg_symbol_length > 0 else 0

        return {
            'avg_comb_length': avg_comb_length,
            'avg_symbol_length': avg_symbol_length,
            'symbol_entropy': symbol_entropy,
            'comb_entropy': comb_entropy,
            'redundancy_comb': redundancy_comb,
            'redundancy_symbol': redundancy_symbol,
            'efficiency_comb': efficiency_comb,
            'efficiency_symbol': efficiency_symbol
        }

def main_combination():
    try:
        n = int(input("Введите количество символов N: "))
        if n <= 0:
            print("Количество символов должно быть положительным!")
            return
    except ValueError:
        print("Ошибка: введите целое число!")
        return

    symbols = []
    probabilities = {}

    print(f"\nВведите {n} символов и их вероятности:")
    for i in range(n):
        while True:
            symbol = input(f"Символ {i+1}: ").strip().upper()
            if not symbol:
                print("Символ не может быть пустым!")
                continue
            if len(symbol) != 1:
                print("Введите один символ!")
                continue
            break

        while True:
            try:
                prob = float(input(f"Вероятность для {symbol}: "))
                if prob <= 0 or prob > 1:
                    print("Вероятность должна быть в диапазоне (0, 1]!")
                    continue
                break
            except ValueError:
                print("Ошибка: введите число!")

        symbols.append(symbol)
        probabilities[symbol] = prob

    total_prob = sum(probabilities.values())
    if abs(total_prob - 1.0) > 0.001:
        print(f"\nПредупреждение: сумма вероятностей = {total_prob:.4f}, нормализуем до 1.0")
        for symbol in probabilities:
            probabilities[symbol] /= total_prob

    try:
        k = int(input(f"\nВведите длину комбинации K: "))
        if k <= 0:
            print("Длина комбинации должна быть положительной!")
            return
        if k > 5:  # Ограничение для больших K
            print(f"Предупреждение: K={k} может создать очень много комбинаций!")
            confirm = input("Продолжить? (y/n): ").lower()
            if confirm != 'y':
                return
    except ValueError:
        print("Ошибка: введите целое число!")
        return

    encoder = ShannonFanoCombination(symbols, probabilities)

    try:
        df, comb_probs = encoder.get_results_table(k)
        stats = encoder.calculate_statistics(comb_probs, k)
        print(f"\nРезультаты кодирования комбинаций:")
        print("=" * 60)
        print(df.to_string(index=False))

        total_combinations = len(df)

        print(f"\nСтатистические характеристики:")
        print("=" * 40)
        print(f"Количество комбинаций: {total_combinations}")
        print(f"Длина комбинации (K): {k}")
        print(f"Средняя длина кода на комбинацию: {stats['avg_comb_length']:.4f}")
        print(f"Средняя длина кода на символ: {stats['avg_symbol_length']:.4f}")
        print(f"Энтропия на символ: {stats['symbol_entropy']:.4f}")
        print(f"Энтропия на комбинацию: {stats['comb_entropy']:.4f}")
        print(f"Избыточность на комбинацию: {stats['redundancy_comb']:.4f}")
        print(f"Избыточность на символ: {stats['redundancy_symbol']:.4f}")

    except MemoryError:
        print(f"Ошибка: слишком много комбинаций для K={k}! Попробуйте меньшее значение K.")
    except Exception as e:
        print(f"Произошла ошибка: {e}")

if __name__ == "__main__":
    main_combination()

Введите количество символов N: 3

Введите 3 символов и их вероятности:
Символ 1: a
Вероятность для A: 0.1
Символ 2: s
Вероятность для S: 0.2
Символ 3: d
Вероятность для D: 0.7

Введите длину комбинации K: 2

Результаты кодирования комбинаций:
Комбинация Вероятность      Код  Длина кода
        DD    0.490000        0           1
        SD    0.140000       10           2
        DS    0.140000      110           3
        AD    0.070000     1110           4
        DA    0.070000    11110           5
        SS    0.040000   111110           6
        AS    0.020000  1111110           7
        SA    0.020000 11111110           8
        AA    0.010000 11111111           8

Статистические характеристики:
Количество комбинаций: 9
Длина комбинации (K): 2
Средняя длина кода на комбинацию: 2.4400
Средняя длина кода на символ: 1.2200
Энтропия на символ: 1.1568
Энтропия на комбинацию: 2.3136
Избыточность на комбинацию: 0.1264
Избыточность на символ: 0.0632


In [ ]:
import pandas as pd
import numpy as np
from itertools import product
from math import log2

class Node:
    def __init__(self, symbol=None, probability=0):
        self.symbol = symbol
        self.probability = probability
        self.left = None
        self.right = None
        self.code = ''

def build_huffman_tree(nodes):
    """Построение дерева Хаффмана"""
    nodes = nodes.copy()

    while len(nodes) > 1:
        nodes.sort(key=lambda x: x.probability)

        left = nodes.pop(0)
        right = nodes.pop(0)

        parent = Node(probability=left.probability + right.probability)
        parent.left = left
        parent.right = right

        nodes.append(parent)

    return nodes[0] if nodes else None

def assign_codes(node, current_code='', codes=None):
    if codes is None:
        codes = {}
    if node is None:
        return codes
    if node.symbol is not None:
        codes[node.symbol] = current_code
        return codes
    assign_codes(node.left, current_code + '1', codes)
    assign_codes(node.right, current_code + '0', codes)
    return codes

def generate_combinations(symbols, probabilities, k):
    """Генерация всех возможных комбинаций длины k"""
    combinations = list(product(symbols, repeat=k))

    comb_probs = []
    for comb in combinations:
        prob = 1.0
        for symbol in comb:
            idx = symbols.index(symbol)
            prob *= probabilities[idx]
        comb_probs.append(prob)

    return combinations, comb_probs

def calculate_statistics(probabilities, codes):
    """Вычисление статистических характеристик"""
    avg_length = sum(prob * len(codes[sym]) for sym, prob in zip(codes.keys(), probabilities))

    entropy = -sum(prob * log2(prob) if prob > 0 else 0 for prob in probabilities)

    redundancy = avg_length - entropy

    efficiency = (entropy / avg_length) * 100 if avg_length > 0 else 0

    return avg_length, entropy, redundancy, efficiency

def huffman_combination_coding():
    """Основная функция для кодирования комбинаций символов"""
    print("Кодирование комбинаций символов методом Хаффмана")
    print("=" * 60)

    try:
        n = int(input("Количество символов (N): "))

        symbols = []
        probabilities = []

        print("\nВведите символы и их вероятности:")
        for i in range(n):
            symbol = input(f"Символ {i+1}: ").strip().upper()
            prob = float(input(f"Вероятность для {symbol}: "))
            symbols.append(symbol)
            probabilities.append(prob)

        k = int(input("Длина последовательности (N): "))

        if n <= 0 or k <= 0:
            raise ValueError("N и K должны быть положительными числами")

        if abs(sum(probabilities) - 1.0) > 1e-10:
            print("Сумма вероятностей не равна 1. Нормализация")
            total = sum(probabilities)
            probabilities = [p/total for p in probabilities]

        combinations, comb_probs = generate_combinations(symbols, probabilities, k)

        nodes = []
        for i, comb in enumerate(combinations):
            comb_str = ''.join(comb)  # Преобразуем кортеж в строку
            nodes.append(Node(comb_str, comb_probs[i]))

        root = build_huffman_tree(nodes)
        codes = assign_codes(root)

        # Вычисление статистических характеристик
        avg_length, entropy, redundancy, efficiency = calculate_statistics(comb_probs, codes)

        results = []
        for comb, prob in zip(combinations, comb_probs):
            comb_str = ''.join(comb)
            results.append({
                'Комбинация': comb_str,
                'Вероятность': f"{prob}",
                'Код': codes[comb_str],
                'Длина кода': len(codes[comb_str])
            })

        df = pd.DataFrame(results)

        df['Вероятность'] = df['Вероятность'].astype(float)
        df = df.sort_values(by='Вероятность', ascending=False).reset_index(drop=True)

        # Вывод результатов
        print(f"\nКоличество символов: {n}")
        print(f"Символы: {symbols}")
        print(f"Вероятности: {[f'{p:}' for p in probabilities]}")
        print(f"Длина комбинации: {k}")
        print(f"Всего комбинаций: {len(combinations)}")

        print("\nТАБЛИЦА КОДОВ:")
        print(df.to_string(index=False))

        print("\nСТАТИСТИЧЕСКИЕ ХАРАКТЕРИСТИКИ:")
        print(f"Средняя длина кода: {avg_length:.4f} бит на комбинацию")
        print(f"Средняя длина на символ: {avg_length/k:.4f} бит на символ")
        print(f"Энтропия: {entropy:.4f} бит на комбинацию")
        print(f"Энтропия на символ: {entropy/k:.4f} бит на символ")
        print(f"Избыточность: {redundancy:.4f} бит")

        return df, avg_length, entropy, redundancy, efficiency

    except Exception as e:
        print(f"Ошибка: {e}")
        return None

if __name__ == "__main__":
    result = huffman_combination_coding()


Кодирование комбинаций символов методом Хаффмана
Количество символов (N): 3

Введите символы и их вероятности:
Символ 1: a
Вероятность для A: 0.1
Символ 2: s
Вероятность для S: 0.2
Символ 3: d
Вероятность для D: 0.7
Длина последовательности (N): 2

Количество символов: 3
Символы: ['A', 'S', 'D']
Вероятности: ['0.1', '0.2', '0.7']
Длина комбинации: 2
Всего комбинаций: 9

ТАБЛИЦА КОДОВ:
Комбинация  Вероятность    Код  Длина кода
        DD         0.49      1           1
        DS         0.14    001           3
        SD         0.14    010           3
        DA         0.07   0000           4
        AD         0.07   0001           4
        SS         0.04   0111           4
        SA         0.02  01101           5
        AS         0.02 011000           6
        AA         0.01 011001           6

СТАТИСТИЧЕСКИЕ ХАРАКТЕРИСТИКИ:
Средняя длина кода: 3.9500 бит на комбинацию
Средняя длина на символ: 1.9750 бит на символ
Энтропия: 2.3136 бит на комбинацию
Энтропия на символ: 1.15